# 次时代：基准评估与价格预测图

这是离线研究 Notebook：读取本地冻结候选样本，比较固定价格规则，并显示真实目标与预测路径。**不联网、不下单、不拟合深度模型。** 默认只展示已保存评估；每次明确新建评估后，后续单元格自动读取并画出这次结果。

先选择可用 Python 3.11+ 内核，再按顺序运行。画图需要 matplotlib；项目采集器本身仍只用标准库。完整说明见 [使用说明](../../docs/usage/FORECAST_DASHBOARD.md)。


In [ ]:
from pathlib import Path
import sys, json
from datetime import datetime, timedelta, timezone
from IPython.display import display, Markdown

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "scripts/build_research_samples.py").is_file()), None)
if PROJECT is None:
    raise RuntimeError("请先在 VS Code 打开 F:/Codex/CS_YouOweMe 文件夹")
sys.dont_write_bytecode = True
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
try:
    import matplotlib.pyplot as plt
except ImportError as error:
    raise RuntimeError("当前内核缺少 matplotlib；请选择已安装 matplotlib 的 Python 内核") from error
from scripts.research.evaluation import baseline_report as evaluator
from scripts.research.visualization import forecast_charts as charts

CHINA = timezone(timedelta(hours=8))
print("项目：", PROJECT)
print("本页只处理本地研究数据，不需要 Cookie 或登录凭证。")


## 1. 读取现有结果，或明确新建一轮离线评估

`RUN_EVALUATION=False`：直接看最新保存结果。需要新版本时设为 `True`，单独运行下面单元格；每轮输出独立目录，旧结果不覆盖。它使用已有候选数据，不会自动抓取最近行情。

规则只在 validation 按 MAE 选择一次，锁定后评估 test；test 展示不能反过来调参。样本源历史可得时间未知，因此是回顾研究，不能认定为当时真实发出的预测或可成交收益。


In [ ]:
RUN_EVALUATION = False
CANDIDATE_ID = None  # None 使用最新候选；也可填已有候选目录编号。
EVALUATION_OUTPUT_ID = None  # None 自动新建版本编号。

if RUN_EVALUATION:
    EVALUATION = evaluator.evaluate(candidate_id=CANDIDATE_ID, output_id=EVALUATION_OUTPUT_ID)
else:
    EVALUATION = evaluator.status()

if not EVALUATION.get("output_dir"):
    raise RuntimeError("还没有基准结果。将 RUN_EVALUATION 改成 True，运行本单元格即可离线生成。")
print("评估版本：", EVALUATION["output_id"])
print("输出目录：", EVALUATION["output_dir"])
print("选定规则：", EVALUATION["selected_model"])
print("实际生成：", EVALUATION.get("generated_at_beijing", "unknown"))


## 2. 先看评估数字

MAE/RMSE 单位为元；误差更小不自动表示可交易。重点比较锁定规则与“不变价”基准在留出期的表现。每日七日目标相互重叠，不把每一行视为独立交易；这里没有费用、价差、实际解锁和退出成交结果。


In [ ]:
TABLES = charts.load_evaluation(EVALUATION)
METRICS = TABLES["metrics"]
display(Markdown("```json\n" + json.dumps(METRICS, ensure_ascii=False, indent=2) + "\n```"))


## 3. 调整显示窗口

`LOOKBACK_DAYS` 控制最近历史图显示 1～28 天。

`FORECAST_ORIGIN_BEIJING=None` 使用评估保存的最近研究起点。也可填候选中已有的完整时点，例如 `"2026-10-12T10:00:00+08:00"`，用同一锁定规则回顾该时点；它不会根据未来价格重选规则。起点与实际生成时刻分别显示。

测试图横轴是七日目标时刻，预测来自该时刻前七天；`TEST_START_TARGET_BEIJING`/`TEST_END_TARGET_BEIJING` 可限制显示范围。缺日期和缺小时断线，不插值。


In [ ]:
LOOKBACK_DAYS = 28
FORECAST_ORIGIN_BEIJING = None
TEST_START_TARGET_BEIJING = None
TEST_END_TARGET_BEIJING = None

def beijing_epoch(value):
    if value is None:
        return None
    dt = datetime.fromisoformat(value)
    if dt.utcoffset() != timedelta(hours=8):
        raise ValueError("显示时点请明确带 +08:00")
    return int(dt.timestamp())

PLOT_OPTIONS = dict(lookback_days=LOOKBACK_DAYS,
                    origin_time=beijing_epoch(FORECAST_ORIGIN_BEIJING),
                    start_target=beijing_epoch(TEST_START_TARGET_BEIJING),
                    end_target=beijing_epoch(TEST_END_TARGET_BEIJING))


## 4. 真实目标与预测路径

第一张图：test 真实七日来源价格、选定规则、不变价基准及误差。

第二张图：研究起点之前的小时收盘，与 day 0～7 的规则日点路径。小时收盘按源 bucket 加一小时绘制，匹配每日时点的信息边界；未来真实价格没有就不画。日点之间的线只是规则推导，不代表小时级波动或置信区间。


In [ ]:
PLOTS = charts.figures(EVALUATION, **PLOT_OPTIONS)
for figure in PLOTS.values():
    display(figure)
    plt.close(figure)

print("来源：SteamDT 显式 YOUPIN 小时参考价格。")
print("预测目标：约168小时后来源参考收盘；不等于解锁后实际出售价格。")
print("当前决策：观察／执行数据不足；没有使用旧截图充当当前报价。")


## 5. 可选：保存这一轮图表

默认只在 Notebook 显示。需要导出时设 `SAVE_CHARTS=True`，图片和追溯清单会写入独立 `data/local/reports/baseline_charts/`，不修改冻结评估结果。保存过的目录不会覆盖。


In [ ]:
SAVE_CHARTS = False
CHART_OUTPUT_ID = None

if SAVE_CHARTS:
    SAVED = charts.render(EVALUATION, output_id=CHART_OUTPUT_ID, **PLOT_OPTIONS)
    print("图表目录：", SAVED["output_dir"])
    for name, path in SAVED["paths"].items():
        print(name, path)
    for figure in SAVED["figures"].values():
        plt.close(figure)
else:
    print("未写出图表；上面的显示仍可查看。")


## 下一次更新

1. 在 SteamDT 采集 Notebook 运行一轮有限 `daily` 更新。
2. 在日级样本 Notebook 明确生成新候选版本。
3. 回到本页，将 `RUN_EVALUATION=True` 新建离线评估，再运行下方展示单元格。

本页不是自动交易程序。UU 当前买卖报价、属性匹配、费用及实际解锁时间需要独立留存并核验；缺失时保留未知，不从来源趋势直接得出买入或保证盈利。
